# Re-run video processing


In [1]:
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import replace
from concurrent.futures import ThreadPoolExecutor, TimeoutError
from time import perf_counter
import shutil

import pandas as pd

from choice_assay.rpi.choice_assay_pose_processor import ChoiceAssayPoseProcessor, ChoiceAssayPoseProcessorCfg, DEFAULT_CHOICE_ASSAY_POSE_PROCESSOR_CFG

Logging expidite to default file: C:\Users\bee-ops\AppData\Local\Temp\expidite\20260628T152047908\logs\default_20260628T152047921.log at level 20
2026-06-28 16:20:47,930 expidite INFO   [19444] Loading C:\Users\bee-ops\.expidite\system.cfg...
Logging choice_assay to default file: C:\Users\bee-ops\AppData\Local\Temp\expidite\20260628T152047908\logs\default_20260628T152047921.log at level 20


In [2]:
# Required config
AZURE_KEYS_FILE = Path.home() / ".expidite" / "keys_choiceassay.env"

CONTAINER_NAME = "expidite-choiceassay-trapcam"
TYPE_ID = "CAVIDEO"

# Local download directory (relative to notebook working directory)
DOWNLOAD_DIR = Path("B://azure/choice_assay/expidite-choiceassay-trapcam")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = Path("B://choice_assay/results/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Fast local cache for JIT staging before ML processing
LOCAL_CACHE_DIR = Path("C:/temp/choice_assay_video_cache")
LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Number of files to keep prefetched ahead of the current inference file
PREFETCH_AHEAD = 6

# Diagnostics for silent waits while staging files from NAS to local cache
PREFETCH_WAIT_LOG_INTERVAL_SECONDS = 30
PREFETCH_HARD_TIMEOUT_SECONDS = None  # e.g. 900 to fail after 15 min

PREFIX = f"V3_{TYPE_ID}_"
SUFFIX = ".mp4"

OUTPUT_PREFIX_LEN = len("V3_CAVIDEO_d83add1a11c5_00_00_20260317")

print(f"Downloading {PREFIX} files from '{CONTAINER_NAME}' to {DOWNLOAD_DIR.resolve()}")
print(f"JIT local cache directory: {LOCAL_CACHE_DIR.resolve()}")

JIT local cache directory: C:\temp\choice_assay_video_cache


In [3]:
# Create a CSV file of all the files in the container.  About 3m for 80k files.
"""
files = DOWNLOAD_DIR.glob(f"{PREFIX}*{SUFFIX}")

# Save the files list to file
files_df = pd.DataFrame([str(f) for f in files], columns=["filename"])
files_df.to_csv("files_list.csv", index=False)
"""

'\nfiles = DOWNLOAD_DIR.glob(f"{PREFIX}*{SUFFIX}")\n\n# Save the files list to file\nfiles_df = pd.DataFrame([str(f) for f in files], columns=["filename"])\nfiles_df.to_csv("files_list.csv", index=False)\n'

In [4]:
def list_processed_videos() -> list[str]:
    """List all the processed video files in the output directory."""
    output_csvs = OUTPUT_DIR.glob(f"{PREFIX}*.csv")

    # Load all the output CSV files and extract the video filenames
    processed_videos = []
    for csv_file in output_csvs:
        df = pd.read_csv(csv_file)
        if "source_filename" in df.columns:
            filenames = df["source_filename"].unique().tolist()
            processed_videos.extend(filenames)
        else:
            print(f"Warning: {csv_file} does not contain 'source_filename' column.")

    return processed_videos

def save_results_to_csv(results: pd.DataFrame, video_fname: str):
    """Save the results to a CSV file."""
    if results.empty:
        print("No results to save.")
        return

    output_csv = OUTPUT_DIR / f"{video_fname[:OUTPUT_PREFIX_LEN]}.csv"
    if not output_csv.exists():
        # Create the output directory if it doesn't exist
        output_csv.parent.mkdir(parents=True, exist_ok=True)

    # Save to CSV: append if exists, otherwise create new
    if output_csv.exists():
        results.to_csv(output_csv, mode='a', header=False, index=False)
    else:
        results.to_csv(output_csv, index=False)

def create_cfg() -> ChoiceAssayPoseProcessorCfg:
    """Create a configuration object for the choice assay pose processor."""
    # we override the sample_probability for the marked up video stream to 0 to avoid generating unnecessary videos during processing
    cfg = DEFAULT_CHOICE_ASSAY_POSE_PROCESSOR_CFG
    marked_up_output = cfg.outputs[1]
    marked_up_output.sample_probability = 0
    return replace(cfg, outputs=[cfg.outputs[0], marked_up_output])

def stage_video_to_local(video_path: Path) -> tuple[Path, float, int, int]:
    """Copy video from NAS to local cache and return path, copy_seconds, remote_size, local_size."""
    local_path = LOCAL_CACHE_DIR / video_path.name
    remote_size = video_path.stat().st_size
    t_copy_start = perf_counter()
    if not local_path.exists():
        shutil.copy2(video_path, local_path)
    copy_seconds = perf_counter() - t_copy_start
    local_size = local_path.stat().st_size
    return local_path, copy_seconds, remote_size, local_size

In [ ]:
# Run ML processing over all the video files in files_list.csv using JIT local staging.
# We prefetch a few files from NAS to local disk so inference reads local files.


# Enable re-run by loading output CSV files and filtering already processed files.
processed_videos = set(list_processed_videos())
videos_to_process = sorted(set(pd.read_csv("files_list.csv")["filename"].to_list()) - processed_videos)

print(f"Found {len(videos_to_process)} new videos to process. {len(processed_videos)} already processed.")
print(f"Prefetch ahead: {PREFETCH_AHEAD} files")

processor = ChoiceAssayPoseProcessor(create_cfg(), 0)
start_time = datetime.now(timezone.utc)

def _submit_prefetch(
    executor: ThreadPoolExecutor,
    queue: dict[int, object],
    submitted_at: dict[int, float],
    index: int,
 ):
    if 0 <= index < len(videos_to_process) and index not in queue:
        remote_path = Path(videos_to_process[index])
        queue[index] = executor.submit(stage_video_to_local, remote_path)
        submitted_at[index] = perf_counter()

def _prefetch_queue_snapshot(
    queue: dict[int, object],
    submitted_at: dict[int, float],
    limit: int = 6,
 ) -> str:
    pending = []
    now = perf_counter()
    for idx in sorted(queue.keys())[:limit]:
        fut = queue[idx]
        age = now - submitted_at.get(idx, now)
        if fut.done():
            state = "done"
        elif fut.running():
            state = "running"
        else:
            state = "pending"
        pending.append(f"{idx}:{state}:{age:.1f}s")
    suffix = " ..." if len(queue) > limit else ""
    return ", ".join(pending) + suffix

prefetch_futures: dict[int, object] = {}
prefetch_submitted_at: dict[int, float] = {}
copy_seconds_total = 0.0
prefetch_wait_seconds_total = 0.0
inference_seconds_total = 0.0

with ThreadPoolExecutor(max_workers=2) as pool:
    # Prime the local staging queue.
    for idx in range(min(PREFETCH_AHEAD, len(videos_to_process))):
        _submit_prefetch(pool, prefetch_futures, prefetch_submitted_at, idx)

    for i in range(len(videos_to_process)):
        _submit_prefetch(pool, prefetch_futures, prefetch_submitted_at, i + PREFETCH_AHEAD)

        remote_video = Path(videos_to_process[i])
        future = prefetch_futures[i]
        wait_start = perf_counter()

        while True:
            try:
                local_video, copy_seconds, remote_size, local_size = future.result(
                    timeout=PREFETCH_WAIT_LOG_INTERVAL_SECONDS
                )
                break
            except TimeoutError:
                wait_so_far = perf_counter() - wait_start
                snapshot = _prefetch_queue_snapshot(prefetch_futures, prefetch_submitted_at)
                print(
                    f"Waiting on prefetch index={i} file={remote_video.name} for {wait_so_far:.1f}s"
                    f" | queue={len(prefetch_futures)} [{snapshot}]"
                )
                if PREFETCH_HARD_TIMEOUT_SECONDS is not None and wait_so_far > PREFETCH_HARD_TIMEOUT_SECONDS:
                    raise TimeoutError(
                        f"Prefetch wait exceeded {PREFETCH_HARD_TIMEOUT_SECONDS}s for {remote_video}"
                    )

        prefetch_wait_seconds_total += perf_counter() - wait_start
        prefetch_futures.pop(i, None)
        prefetch_submitted_at.pop(i, None)
        copy_seconds_total += copy_seconds

        assert local_video.exists(), f"Local video {local_video} does not exist after prefetching."
        assert local_video.name == remote_video.name, f"Local video {local_video} name does not match remote video {remote_video.name}"

        # Process local file; keep original filename for output naming.
        t_infer_start = perf_counter()
        df = processor._process_video_file(local_video)
        inference_seconds = perf_counter() - t_infer_start
        inference_seconds_total += inference_seconds
        save_results_to_csv(df, remote_video.name)

        # Keep cache size bounded by removing files right after processing.
        try:
            local_video.unlink(missing_ok=True)
        except OSError as err:
            print(f"Warning: failed to remove cache file {local_video}: {err}")

        elapsed = (datetime.now(timezone.utc) - start_time).total_seconds()
        n = i + 1
        print(
            f"{n}/{len(videos_to_process)} @ {elapsed / n:.1f} secs/video"
            f" | infer={inference_seconds:.2f}s"
            f" | copy={copy_seconds:.2f}s ({remote_size / 1e6:.1f}MB)"
            f" | avg_copy={copy_seconds_total / n:.2f}s"
            f" | avg_wait={prefetch_wait_seconds_total / n:.2f}s"
            f" | avg_infer={inference_seconds_total / n:.2f}s"
            f" | cache_queue={len(prefetch_futures)}"
            f" | processed {remote_video.name} | output -> {OUTPUT_DIR.resolve()}"
        )

Found 81358 new videos to process. 343 already processed.
Prefetch ahead: 6 files
2026-06-28 16:20:58,273 choice_assay INFO   [19444] Pose model diagnostics: ultralytics=8.4.21 model_path=C:\Users\bee-ops\code\ChoiceAssay\src\choice_assay\resources\best.pt names_count=2 names_keys=[0, 1]
2026-06-28 16:20:58,533 choice_assay INFO   [19444] Pose frame diagnostics: video=C:\temp\choice_assay_video_cache\V3_CAVIDEO_d83add1a11c5_00_00_20260506T133941023_20260506T134001223.mp4 frame=0 detected_classes=[1] missing_name_ids=[]
2026-06-28 16:21:06,101 choice_assay INFO   [19444] Pose timings: video=C:\temp\choice_assay_video_cache\V3_CAVIDEO_d83add1a11c5_00_00_20260506T133941023_20260506T134001223.mp4 model_load=0.074s predict_setup=0.094s stream_iter=7.731s dataframe=0.002s markup_save=0.000s total=7.902s frames=102 rows=102 rows_per_frame=1.000
1/81358 @ 8.7 secs/video | infer=7.91s | copy=0.58s (0.4MB) | avg_copy=0.58s | avg_wait=0.68s | avg_infer=7.91s | cache_queue=6 | processed V3_CAVIDEO

In [ ]:
"""
# Delete all the files in the old_files_list.csv from the container
old_files_df = pd.read_csv("old_files_list.csv")

cc = CloudConnector.get_instance(root_cfg.CloudType.AZURE)
cc.set_keys(AZURE_KEYS_FILE)
old_files = old_files_df["filename"].tolist()

# To efficiently delete files, we want to pass batches directly to the appropriate Azure API.
# The CloudConnector class doesn't have a method for this
containerClient = cc._validate_container(CONTAINER_NAME)

# We process in batches of 256 files at a time, as this is the maximum number of files that can be deleted in a
# single request to the Azure API.
for i in range(0, len(old_files), 256):
    batch = old_files[i:i + 256]
    containerClient.delete_blobs(*batch, raise_on_any_failure=False)
    print(f"{i} - Deleted {len(batch)} files from container '{CONTAINER_NAME}'")
"""

In [ ]:
csv_paths = sorted(DOWNLOAD_DIR.rglob("*.csv"))
print(f"CSV files available locally: {len(csv_paths)}")

df_list = []
for csv_path in csv_paths:
    df = pd.read_csv(csv_path)
    if not df.empty:
        df["source_file"] = csv_path.name
        df_list.append(df)

if df_list:
    aggregated_df = pd.concat(df_list, ignore_index=True)
else:
    aggregated_df = pd.DataFrame()

print(f"Aggregated rows: {len(aggregated_df)}")
aggregated_df.head()

In [ ]:
# Data validation before behaviour classification
required_columns = ["Tube_prob_likelihood", "End_prob_likelihood"]
missing_columns = [col for col in required_columns if col not in aggregated_df.columns]

if aggregated_df.empty:
    print("Validation: aggregated_df is empty.")
elif missing_columns:
    msg = f"Validation failed. Missing required columns: {missing_columns}"
    raise KeyError(msg)
else:
    for col in required_columns:
        aggregated_df[col] = pd.to_numeric(aggregated_df[col], errors="coerce")

    invalid_rows = aggregated_df[required_columns].isna().any(axis=1).sum()
    print(f"Validation: {len(aggregated_df)} total rows")
    print(f"Validation: {invalid_rows} rows have invalid/missing likelihood values")

    if invalid_rows:
        print(
            aggregated_df.loc[
                aggregated_df[required_columns].isna().any(axis=1), [*required_columns, "source_file"]
            ].head()
        )

In [ ]:
# Define the function to classify behavior
def get_behaviour(row: pd.Series) -> str:
    behaviour = "No_prob"
    if (row["Tube_prob_likelihood"] >= 0.6) & (row["End_prob_likelihood"] >= 0.6):
        behaviour = "Drinking"
    elif (row["Tube_prob_likelihood"] >= 0.6) ^ (row["End_prob_likelihood"] >= 0.6):
        behaviour = "Prob_out"
    return behaviour


assert aggregated_df is not None, "aggregated_df should be defined at this point"
required_columns = ["Tube_prob_likelihood", "End_prob_likelihood"]
missing_columns = [col for col in required_columns if col not in aggregated_df.columns]

if aggregated_df.empty:
    print("No data loaded; behaviour classification skipped.")
elif missing_columns:
    msg = f"Missing required columns for behaviour classification: {missing_columns}"
    raise KeyError(msg)
else:
    before_count = len(aggregated_df)
    clean_df = aggregated_df.dropna(subset=required_columns).copy()
    dropped_count = before_count - len(clean_df)

    if dropped_count:
        print(f"Dropped {dropped_count} rows with invalid/missing likelihood values before classification.")

    clean_df["Behaviour"] = clean_df.apply(get_behaviour, axis=1)
    aggregated_df = clean_df
    print(aggregated_df["Behaviour"].value_counts(dropna=False))

aggregated_df.head()

In [ ]:
output_path = DOWNLOAD_DIR / "aggregated_journals_with_behaviour.csv"
if not aggregated_df.empty:
    aggregated_df.to_csv(output_path, index=False)
    print(f"Saved aggregated dataset to: {output_path.resolve()}")
else:
    print("Aggregated dataframe is empty; no output written.")